# Aula 02 — Grafos: o básico que tudo o resto usa

**Objetivo:** montar, rodar e principalmente **quebrar** um grafo de propósito, até
entender o que cada peça faz de verdade. Ao final você deve saber ler um erro de
rota, saber por que uma etapa não rodou e saber o que é `overwrite`.

**Pré-requisito:** Aula 00 (você já sabe o que é Estado, Nó, Aresta) e Aula 01 (ambiente).

**Como usar:** rode em ordem. Autocontido (funciona no Colab com 1 arquivo).
Espelho local: `python src/aula02_grafo.py`. Todos os comportamentos aqui foram
**testados no LangGraph 1.2.x** — inclusive as surpresas da seção 5.

## 1. Analogia: o relatório do lanche como dict + função + `if`

Pense num atendimento de lanchonete: um **dict de pedido** vai sendo preenchido por
funções, e um `if` escolhe a bebida. O LangGraph repete exatamente essa forma, com
nomes novos:

| Você já conhece | No LangGraph chama | O que é de verdade |
|---|---|---|
| dict do pedido | **Estado** | `TypedDict` — só um contrato de chaves |
| função que devolve o que mudou | **Nó** | `f(state) -> dict parcial` |
| chamada em sequência | **Aresta** | ordem fixa `A -> B` |
| `if/else` | **Aresta condicional** | função que devolve o **nome** do próximo nó |
| fluxograma montado | **`compile()`** | valida o desenho e vira executável |
| rodar com um input | **`invoke(input)`** | executa do `START` ao `END` |

Traduzindo o lanche: `receber` adiciona hambúrguer; `decide_bebida` lê `com_sede`
e devolve o **nome** do nó seguinte (`com_sede` ou `sem_sede`); cada nó de bebida
anexa seu item. Nada além de função + dict + if.

In [1]:
# CÉLULA 1 — o lanche no LangGraph (revisão da Aula 00)
from typing import TypedDict
from langgraph.graph import START, END, StateGraph

class Pedido(TypedDict):
    itens: list
    com_sede: bool

def receber(state: Pedido) -> dict:
    return {'itens': state['itens'] + ['hamburguer']}

def com_sede(state: Pedido) -> dict:
    return {'itens': state['itens'] + ['suco']}

def sem_sede(state: Pedido) -> dict:
    return {'itens': state['itens'] + ['agua']}

def decide_bebida(state: Pedido) -> str:
    return 'com_sede' if state['com_sede'] else 'sem_sede'

builder = StateGraph(Pedido)
builder.add_node('receber', receber)
builder.add_node('com_sede', com_sede)
builder.add_node('sem_sede', sem_sede)
builder.add_edge(START, 'receber')
builder.add_conditional_edges('receber', decide_bebida)   # o if, como roteador
builder.add_edge('com_sede', END)
builder.add_edge('sem_sede', END)
grafo_lanche = builder.compile()

print(grafo_lanche.invoke({'itens': [], 'com_sede': True})['itens'])
print(grafo_lanche.invoke({'itens': [], 'com_sede': False})['itens'])
assert grafo_lanche.invoke({'itens': [], 'com_sede': True})['itens'] == ['hamburguer', 'suco']
print('OK — bifurcação funcionando.')

['hamburguer', 'suco']
['hamburguer', 'agua']
OK — bifurcação funcionando.


## 2. Conceito: o que é real e o que é só aparência

- **`class State(TypedDict)`** é um **contrato de leitura**, não validação. Testado:
  passar uma chave a mais no `invoke` não dá erro; passar uma a **menos** só quebra
  quando um nó tenta lê-la (`KeyError`, seção 5d).
- **`Literal["node_2", "node_3"]` no retorno do roteador é só hint** — escrever
  `-> str` e devolver a mesma string funciona igual. Quem decide a rota é a
  **string retornada**, não a anotação de tipo.
- **O nó devolve update parcial**, nunca o estado inteiro. E, muito importante:
  **quem não devolve não persiste** (seção 5e).
- **`add_node("nome", funcao)`** registra a função sob um nome. As arestas falam
  só com **nomes**. Aresta = `add_edge("de", "para")`.
- **`START` e `END`** são marcadores especiais (nunca são nós seus).

O mesmo grafo dos 3 nós do curso, com uma condicional 50/50 — mas com `seed`,
para o estudo ser **reprodutível**:

In [ ]:
# CÉLULA 2 — o grafo de 3 nós (com seed para estudo)
import random
from typing import TypedDict
from langgraph.graph import START, END, StateGraph

class State(TypedDict):
    graph_state: str

def node_1(state: State) -> dict:
    return {'graph_state': state['graph_state'] + ' Eu estou '}

def node_2(state: State) -> dict:
    return {'graph_state': state['graph_state'] + 'cansado!'}

def node_3(state: State) -> dict:
    return {'graph_state': state['graph_state'] + 'descansado!'}

def decide_mood(state: State) -> str:      # str simples: Literal é só hint
    return 'node_2' if random.random() < 0.5 else 'node_3'

builder = StateGraph(State)
builder.add_node('node_1', node_1)
builder.add_node('node_2', node_2)
builder.add_node('node_3', node_3)
builder.add_edge(START, 'node_1')
builder.add_conditional_edges('node_1', decide_mood)     # decide o caminho em runtime
builder.add_edge('node_2', END)
builder.add_edge('node_3', END)
grafo = builder.compile()

for seed in (1, 2, 3, 4):
    random.seed(seed)                                    # mesma seed => mesmo caminho
    print(f'seed={seed} ->', grafo.invoke({'graph_state': 'Oi,'})['graph_state'])

## 3. Dissecando a célula 2 linha a linha

- `class State(TypedDict): graph_state: str` — o único "canal" deste grafo.
- `node_1(state) -> dict` — recebe o estado, devolve **só** `{'graph_state': ...}`.
- `decide_mood(state)` — **roteador**: devolve `'node_2'` ou `'node_3'`.
  O valor retornado tem que ser **exatamente igual** ao nome registrado no `add_node`.
- `builder = StateGraph(State)` — começamos o rascunho tipado.
- `builder.add_node('node_1', node_1)` — nome + função (não precisa ser o mesmo nome).
- `builder.add_edge(START, 'node_1')` — sempre começa aqui.
- `builder.add_conditional_edges('node_1', decide_mood)` — após `node_1`, pergunte
  ao roteador e vá para onde ele mandar.
- `builder.add_edge('node_2', END)` / `('node_3', END)` — os dois caminhos terminam.
- `builder.compile()` — valida o desenho. Sem ele, não existe grafo executável.
- `random.seed(seed)` antes de cada `invoke` — didático: assim você vê os dois
  caminhos sem depender da sorte (em produção, sem seed o LLM é que decide).
- `graph.invoke({'graph_state': 'Oi,'})` — **uma execução** inteira, devolve o
  estado final.

## 4. Variação A — grafo linear (sem condicional)

O mesmo estado em sequência fixa: `node_1 -> node_2 -> node_3`. Repare que não há
decisão nenhuma: três arestas fixas, três concatenações.

In [ ]:
# CÉLULA 3 — versão linear do mesmo grafo
from typing import TypedDict
from langgraph.graph import START, END, StateGraph

class StateLinear(TypedDict):
    graph_state: str

def a(state): return {'graph_state': state['graph_state'] + ' Eu estou '}
def b(state): return {'graph_state': state['graph_state'] + 'cansado'}
def c(state): return {'graph_state': state['graph_state'] + '!'}

bl = StateGraph(StateLinear)
bl.add_node('node_1', a)
bl.add_node('node_2', b)
bl.add_node('node_3', c)
bl.add_edge(START, 'node_1')
bl.add_edge('node_1', 'node_2')
bl.add_edge('node_2', 'node_3')
bl.add_edge('node_3', END)
linear = bl.compile()

print(linear.invoke({'graph_state': 'Oi,'})['graph_state'])
assert linear.invoke({'graph_state': 'Oi,'})['graph_state'] == 'Oi, Eu estou cansado!'
print('OK — reta: START -> 1 -> 2 -> 3 -> END.')

## 5. Variações que quebram (os comportamentos que ninguém te conta)

Todos os itens abaixo foram **executados neste ambiente** (LangGraph 1.2.x).
Leia antes de rodar: alguns não dão erro — e esses são os piores.

### 5a. `overwrite`: dois nós na mesma chave → o **último** ganha (sem erro)

Se `A` escreve `graph_state` e `B` escreve depois, o valor de `A` desaparece.
Esse é o padrão do LangGraph e é o que vai quebrar na Aula 08 quando os dois
rodarem **em paralelo** (aí sim dá `InvalidUpdateError`).

In [ ]:
# CÉLULA 4 — overwrite em sequência
from typing import TypedDict
from langgraph.graph import START, END, StateGraph

class StateOv(TypedDict):
    graph_state: str

def no_a(state: StateOv) -> dict: return {'graph_state': 'escrito por A'}
def no_b(state: StateOv) -> dict: return {'graph_state': 'escrito por B'}

bo = StateGraph(StateOv)
bo.add_node('a', no_a); bo.add_node('b', no_b)
bo.add_edge(START, 'a'); bo.add_edge('a', 'b'); bo.add_edge('b', END)
print(bo.compile().invoke({'graph_state': 'inicial'}))
# esperado: {'graph_state': 'escrito por B'}  -> o valor de A e o inicial sumiram

### 5b. Rota digitada errada **não quebra** — só não executa a etapa

Pegadinha grande: se o roteador devolve um nome que **não existe**, o LangGraph
emite um **aviso** e o grafo encerra silenciosamente. Nenhum erro no seu código.
Você só descobre que `node_2`/`node_3` nunca rodaram quando olha o resultado:
o sufixo `cansado!` nunca aparece. Em notebook o aviso aparece como saída do
kernel — aprenda a ler.

In [ ]:
# CÉLULA 5 — rota inexistente: aviso + encerramento (sem exceção!)
from typing import TypedDict
from langgraph.graph import START, StateGraph

class StateQ(TypedDict):
    graph_state: str

def node_1(state: StateQ) -> dict:
    return {'graph_state': state['graph_state'] + ' Eu estou '}

def rota_quebrada(state: StateQ) -> str:
    return 'node_99'          # nome que NÃO existe no grafo

bq = StateGraph(StateQ)
bq.add_node('node_1', node_1)
bq.add_edge(START, 'node_1')
bq.add_conditional_edges('node_1', rota_quebrada)
print(bq.compile().invoke({'graph_state': 'Oi,'}))
# observe: sem 'cansado!' -> node_2 e node_3 NUNCA rodaram.
# o aviso 'wrote to unknown channel branch:to:node_99' confirma.

### 5c. Chave fora do schema é **ignorada** em silêncio

Atualização com chave que não está no `TypedDict` é descartada — estado intacto.
Ou seja, `TypedDict` não protege seu estado; ele só documenta.

In [ ]:
# CÉLULA 6 — chave fora do schema
from typing import TypedDict
from langgraph.graph import START, END, StateGraph

class StateS(TypedDict):
    graph_state: str

def inventor(state: StateS) -> dict:
    return {'nao_existo': 123}        # fora do contrato

bs = StateGraph(StateS)
bs.add_node('inventor', inventor)
bs.add_edge(START, 'inventor'); bs.add_edge('inventor', END)
print(bs.compile().invoke({'graph_state': 'valor original'}))
# esperado: {'graph_state': 'valor original'} -> a chave inventada sumiu

### 5d. Input incompleto: o erro só aparece quando **um nó lê** a chave

O `invoke` não valida o estado contra o schema. Se faltar `graph_state`, o grafo
roda até o primeiro nó que tentar ler, e aí estoura `KeyError`.

In [ ]:
# CÉLULA 7 — input incompleto: KeyError só quando UM NÓ lê a chave
from typing import TypedDict
from langgraph.graph import START, END, StateGraph

class StateI(TypedDict):
    graph_state: str

def le_estado(state: StateI) -> dict:
    return {'graph_state': state['graph_state'] + '!'}   # lê state['graph_state']

bi = StateGraph(StateI)
bi.add_node('le', le_estado)
bi.add_edge(START, 'le'); bi.add_edge('le', END)

try:
    print(bi.compile().invoke({'outra_chave': 'x'}))
except KeyError as erro:
    print('KeyError:', erro, '-> faltou a chave que o nó precisava ler')

### 5e. Nó que só **muta** e não devolve nada → a mudança **não persiste**

Erro clássico de quem pensa em classe: `state['graph_state'] += '!'` parece que
funcionou, mas como o nó devolveu `{}`, o estado seguiu igual.

In [ ]:
# CÉLULA 8 — mutação sem retorno
from typing import TypedDict
from langgraph.graph import START, END, StateGraph

class StateM(TypedDict):
    graph_state: str

def so_muta(state: StateM) -> dict:
    state['graph_state'] += '!'      # muta localmente...
    return {}                        # ...mas não devolve nada

bm = StateGraph(StateM)
bm.add_node('m', so_muta)
bm.add_edge(START, 'm'); bm.add_edge('m', END)
print(bm.compile().invoke({'graph_state': 'oi'}))
# esperado: {'graph_state': 'oi'} -> SEM '!' porque o update não voltou no return

### 5f. Erros que aparecem na construção (antes de rodar)

- Aresta para nó inexistente → `ValueError: Found edge ending at unknown node`
- Nó registrado duas vezes → `ValueError: Node 'x' already present`
- Grafo sem aresta a partir do `START` → `ValueError: Graph must have an entrypoint`
- Função de nó sem o parâmetro de estado → `TypeError` no `invoke`

In [ ]:
# CÉLULA 9 — erros de construção
from typing import TypedDict
from langgraph.graph import START, END, StateGraph

class StateE(TypedDict):
    graph_state: str

def no(state: StateE) -> dict:
    return {'graph_state': state['graph_state'] + 'x'}

# 1) aresta para nó fantasma
b1 = StateGraph(StateE); b1.add_node('node_1', no); b1.add_edge(START, 'node_1')
try:
    b1.add_edge('node_1', 'fantasma')
except ValueError as e:
    print('aresta fantasma ->', e)

# 2) nó duplicado
b2 = StateGraph(StateE); b2.add_node('node_1', no)
try:
    b2.add_node('node_1', no)
except ValueError as e:
    print('nó duplicado ->', e)

# 3) sem entrypoint (nenhuma aresta a partir do START)
b3 = StateGraph(StateE); b3.add_node('node_1', no)
try:
    b3.compile()
except ValueError as e:
    print('sem entrypoint ->', e)

# 4) nó sem argumento
def sem_arg():
    return {}
b4 = StateGraph(StateE); b4.add_node('x', sem_arg)
b4.add_edge(START, 'x'); b4.add_edge('x', END)
try:
    b4.compile().invoke({'graph_state': 'x'})
except TypeError as e:
    print('nó sem arg ->', e)

### 5g. Visualizando o grafo: `draw_mermaid` sempre, `draw_ascii` se der

`draw_mermaid()` devolve o **texto** do diagrama (cole no GitHub, no Notion ou no
[mermaid.live](https://mermaid.live) para ver a figura). `draw_ascii()` depende da
biblioteca `grandalf`, que **não** vem instalada — por isso o `try/except`: aprenda
o padrão "dependência opcional degrada com elegância".

In [ ]:
# CÉLULA 10 — desenhando o grafo de 3 nós (usa o `grafo` da CÉLULA 2)
print('--- draw_mermaid() ---')
print(grafo.get_graph().draw_mermaid())

print('--- draw_ascii() (opcional) ---')
try:
    print(grafo.get_graph().draw_ascii())
except ImportError as erro:
    print('ASCII indisponível:', erro)
    print('para ter: pip install grandalf')

## 6. Erros clássicos de quem vem do determinístico

1. **Contar que o nó "atribuiu" no estado sem devolver** (5e). Atualização só
   existe no `return` do nó.
2. **Achar que `TypedDict` valida** (5c/5d). Ele documenta; a validação real é
   Pydantic, na Aula 07.
3. **Achar que rota errada dá erro** (5b). Dá aviso e encerra. Leia o aviso.
4. **Esperar que o nó saiba o "próximo passo"** — quem decide a rota é a aresta
   condicional. `if` dentro do nó só muda dados, não caminho.
5. **Esperar que nós paralelos escrevam na mesma chave** — em sequência o último
   ganha (5a), em paralelo dá `InvalidUpdateError` (Aula 08, com `reducer`).
6. **Confundir chave do estado com nome do nó.** O roteador devolve **nome de nó**;
   `return {'graph_state': ...}` devolve **chave de estado**. Trocar os dois é o
   bug mais comum de quem está aprendendo.

## 7. Exercício guiado — lanchonete com 3 rotas

Complete o código abaixo: `base` anexa `hamburguer`; `decide` escolhe
`com_sede`/`sem_sede`/`vegetariano`; cada nó anexa seu item. Os `assert`
no final validam os três caminhos.

In [ ]:
# CÉLULA 11 — exercício guiado (complete as linhas marcadas)
from typing import TypedDict
from langgraph.graph import START, END, StateGraph

class Pedido3(TypedDict):
    itens: list
    bebida: str        # 'suco' | 'agua' | 'nenhuma'

def base(state: Pedido3) -> dict:
    return {'itens': state['itens'] + ['hamburguer']}

def com_sede(state: Pedido3) -> dict:
    return {'itens': state['itens'] + ['suco'], 'bebida': 'suco'}

def sem_sede(state: Pedido3) -> dict:
    return {'bebida': 'agua'}

def vegetariano(state: Pedido3) -> dict:
    return {'itens': state['itens'] + ['salada'], 'bebida': 'nenhuma'}

def decide(state: Pedido3) -> str:
    if state['bebida'] == 'suco':
        return 'com_sede'          # complete: nome do nó
    if state['bebida'] == 'agua':
        return 'sem_sede'
    return 'vegetariano'

b = StateGraph(Pedido3)
b.add_node('base', base)
b.add_node('com_sede', com_sede)
b.add_node('sem_sede', sem_sede)
b.add_node('vegetariano', vegetariano)
b.add_edge(START, 'base')
b.add_conditional_edges('base', decide)   # o if virou roteador
b.add_edge('com_sede', END)
b.add_edge('sem_sede', END)
b.add_edge('vegetariano', END)
g3 = b.compile()

r1 = g3.invoke({'itens': [], 'bebida': 'suco'})
r2 = g3.invoke({'itens': [], 'bebida': 'agua'})
r3 = g3.invoke({'itens': [], 'bebida': 'nenhuma'})
print(r1, '|', r2, '|', r3)

assert r1 == {'itens': ['hamburguer', 'suco'], 'bebida': 'suco'}, r1
assert r2 == {'itens': ['hamburguer'], 'bebida': 'agua'}, r2
assert r3 == {'itens': ['hamburguer', 'salada'], 'bebida': 'nenhuma'}, r3
print('OK — 3 rotas, 1 roteador.')

## 8. Desafio — avaliador de nota (2 nós, 3 canais)

Situação: um grafo que recebe `aluno` e `nota`, decide `aprovado`/`reprovado`
num nó e grava o `status` no estado. É a semente do que vocês já fizeram no
avaliador de redação — e vai direcionar a Aula 07 (schemas com `Literal`).

**Regras:**
- Canais: `aluno` (str), `nota` (int), `status` (str)
- `calcular`: nota ≥ 7 → `status='aprovado'`, senão `status='reprovado'`
- `historico`: devolve um texto explicando a decisão
- Rota final: sempre `END`

**Dica:** use `decide` para separar os dois caminhos, e grave o `status`
dentro de cada nó — não no roteador (roteador só escolhe caminho).

In [ ]:
# CÉLULA 12 — desafio avaliador de nota (aqui com uma versão pronta para comparar)
from typing import TypedDict
from langgraph.graph import START, END, StateGraph

class Resultado(TypedDict):
    aluno: str
    nota: int
    status: str

def aprovado(state: Resultado) -> dict:
    return {'status': 'aprovado', 'nota': state['nota']}

def reprovado(state: Resultado) -> dict:
    return {'status': 'reprovado', 'nota': state['nota']}

def decide(state: Resultado) -> str:
    # roteador: SÓ escolhe o caminho (não grava nada no estado)
    return 'aprovado' if state['nota'] >= 7 else 'reprovado'

b = StateGraph(Resultado)
b.add_node('aprovado', aprovado)
b.add_node('reprovado', reprovado)
b.add_conditional_edges(START, decide)   # teste: a condicional pode sair do START!
b.add_edge('aprovado', END)
b.add_edge('reprovado', END)
avaliador = b.compile()

r1 = avaliador.invoke({'aluno': 'Ana', 'nota': 8, 'status': ''})
r2 = avaliador.invoke({'aluno': 'Bia', 'nota': 5, 'status': ''})
print(r1['aluno'], r1['status'], '|', r2['aluno'], r2['status'])

assert r1['status'] == 'aprovado', r1
assert r2['status'] == 'reprovado', r2
print('OK — note que o `status` veio do NÓ escolhido, não do roteador.')

# DESAFIO EXTRA (sem solução aqui): adicione um nó `recuperacao` para nota entre 5 e 7
# e use 3 rotas no `decide`.

## 9. Checklist + próxima aula

**Você entendeu se consegue responder sem olhar:**
- [ ] O que o nó **devolve** e por que mutar o estado sem retornar não funciona?
- [ ] Como uma aresta condicional escolhe o caminho (e o que o retorno significa)?
- [ ] Se o roteador devolve um nome inexistente, acontece erro ou silêncio?
- [ ] Em sequência, quando dois nós escrevem a mesma chave, quem ganha?
- [ ] Qual a diferença entre **chave do estado** e **nome de nó**?
- [ ] Para que servem `compile()` e `invoke()`, e o que `START`/`END` representam?

**Próxima aula (03 — Cadeias):** estado como lista de mensagens, `bind_tools`
(o LLM pede uma ferramenta sem executá-la) e `add_messages` — o passo anterior
ao agente reAct da Aula 05.